# Task 2. 누적 적재 데이터 최신성·흐름 검증

이 노트북은 현재 Python source code가 정의한 raw Delta 계약과 실제 `data/delta/ethereum_logs`, `data/analytics/ethereum_analytics.duckdb` 산출물이 일치하는지 확인합니다.

검증 목적은 단순 row count 확인이 아닙니다. 평가자가 코드와 데이터의 최신성, raw → dbt silver → dbt gold 흐름, 중복 방지 상태, stale schema 여부를 한 번에 확인할 수 있도록 만드는 데 있습니다.

외부 RPC Provider는 호출하지 않습니다. 이미 적재된 로컬 산출물과 fixture 기반 검증 결과를 기준으로 판단합니다.

## 운영 규칙

- canonical raw Delta 경로는 `data/delta/ethereum_logs`입니다.
- canonical DuckDB 경로는 `data/analytics/ethereum_analytics.duckdb`입니다.
- Docker 기본 경로는 `/opt/airflow/data/delta/ethereum_logs`, `/opt/airflow/data/analytics/ethereum_analytics.duckdb`입니다.
- `_v2` suffix가 붙은 legacy 경로는 현재 제출 기준으로 사용하지 않습니다.
- `DELTA_LOGS_PATH`, `DUCKDB_PATH`는 Docker Compose가 stale `.env`에서 주입할 수 있으므로 이 노트북에서는 직접 사용하지 않습니다.
- 필요하면 `NOTEBOOK_DELTA_LOGS_PATH`, `NOTEBOOK_DUCKDB_PATH`만 명시적으로 지정합니다.

## 0. import 및 helper

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Any

import duckdb
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable
from IPython.display import Markdown, display

DISPLAY_TZ = "Asia/Seoul"


def find_project_root(start: Path | None = None) -> Path:
    """현재 노트북 위치와 실행 cwd가 달라도 repository root를 찾는다."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "cryptoquant_pipeline").exists():
            return candidate
    raise RuntimeError("pyproject.toml과 src/cryptoquant_pipeline을 가진 repository root를 찾지 못했다.")


PROJECT_ROOT = find_project_root()
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cryptoquant_pipeline.delta_writer import (  # noqa: E402
    NATURAL_KEY_COLUMNS,
    PARTITION_COLUMNS,
    ethereum_logs_schema,
)


def first_existing_path(candidates: list[Path], *, label: str) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    formatted = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(f"{label} 경로를 찾지 못했다. 후보:\n{formatted}")


def optional_env_path(name: str) -> list[Path]:
    value = os.environ.get(name)
    return [Path(value)] if value else []


DELTA_LOGS_PATH = first_existing_path(
    [
        *optional_env_path("NOTEBOOK_DELTA_LOGS_PATH"),
        PROJECT_ROOT / "data" / "delta" / "ethereum_logs",
        Path("/workspace/data/delta/ethereum_logs"),
        Path("/opt/airflow/data/delta/ethereum_logs"),
    ],
    label="raw Delta ethereum_logs",
)
DUCKDB_PATH = first_existing_path(
    [
        *optional_env_path("NOTEBOOK_DUCKDB_PATH"),
        PROJECT_ROOT / "data" / "analytics" / "ethereum_analytics.duckdb",
        Path("/workspace/data/analytics/ethereum_analytics.duckdb"),
        Path("/opt/airflow/data/analytics/ethereum_analytics.duckdb"),
    ],
    label="DuckDB analytics",
)


def as_utc_timestamp(value: Any) -> pd.Timestamp | None:
    if value is None or pd.isna(value):
        return None
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        return ts.tz_localize("UTC")
    return ts.tz_convert("UTC")


def add_kst_column(df: pd.DataFrame, utc_column: str, kst_column: str | None = None) -> pd.DataFrame:
    if df.empty or utc_column not in df.columns:
        return df
    result = df.copy()
    target = kst_column or utc_column.replace("_utc", "_kst")
    result[target] = pd.to_datetime(result[utc_column], utc=True, errors="coerce").dt.tz_convert(DISPLAY_TZ)
    return result


def safe_execute_df(con: duckdb.DuckDBPyConnection, sql: str, params: list[Any] | None = None) -> tuple[pd.DataFrame | None, str | None]:
    try:
        return con.execute(sql, params or []).fetchdf(), None
    except Exception as exc:  # notebook 검증 셀은 실패 내용을 표로 남겨야 함
        return None, f"{type(exc).__name__}: {exc}"


def safe_scalar(con: duckdb.DuckDBPyConnection, sql: str, params: list[Any] | None = None) -> tuple[Any | None, str | None]:
    try:
        return con.execute(sql, params or []).fetchone()[0], None
    except Exception as exc:  # notebook 검증 셀은 실패 내용을 표로 남겨야 함
        return None, f"{type(exc).__name__}: {exc}"


display(Markdown(
    "### 검증 대상 경로\n"
    "```text\n"
    f"project_root = {PROJECT_ROOT}\n"
    f"delta_logs_path = {DELTA_LOGS_PATH}\n"
    f"duckdb_path = {DUCKDB_PATH}\n"
    "```"
))

### 검증 대상 경로
```text
project_root = /workspace
delta_logs_path = /workspace/data/delta/ethereum_logs
duckdb_path = /workspace/data/analytics/ethereum_analytics.duckdb
```

## 1. Python source raw 계약과 실제 Delta schema 비교

현재 raw schema의 정본은 `src/cryptoquant_pipeline/delta_writer.py`의 `ethereum_logs_schema()`입니다. 이 셀은 노트북 내부 하드코딩 대신 Python source를 직접 import해서 실제 Delta table schema와 비교합니다.

In [2]:
delta_table = DeltaTable(str(DELTA_LOGS_PATH))
actual_schema = delta_table.schema().to_pyarrow()
expected_schema = ethereum_logs_schema(pa)

actual_by_name = {field.name: field for field in actual_schema}
expected_by_name = {field.name: field for field in expected_schema}
all_columns = sorted(set(actual_by_name) | set(expected_by_name))

schema_rows: list[dict[str, Any]] = []
for column_name in all_columns:
    expected = expected_by_name.get(column_name)
    actual = actual_by_name.get(column_name)
    if expected is None:
        status = "EXTRA_IN_DATA"
    elif actual is None:
        status = "MISSING_IN_DATA"
    elif str(expected.type) != str(actual.type):
        status = "TYPE_MISMATCH"
    elif expected.nullable != actual.nullable:
        status = "NULLABILITY_MISMATCH"
    else:
        status = "PASS"

    schema_rows.append(
        {
            "column_name": column_name,
            "expected_type": str(expected.type) if expected else None,
            "actual_type": str(actual.type) if actual else None,
            "expected_nullable": expected.nullable if expected else None,
            "actual_nullable": actual.nullable if actual else None,
            "status": status,
        }
    )

schema_contract_df = pd.DataFrame(schema_rows)
blocking_schema_statuses = {"MISSING_IN_DATA", "TYPE_MISMATCH", "NULLABILITY_MISMATCH"}
RAW_SCHEMA_CURRENT = not schema_contract_df["status"].isin(blocking_schema_statuses).any()

status_text = "VERIFIED" if RAW_SCHEMA_CURRENT else "PARTIALLY VERIFIED"
display(Markdown(
    "### Raw schema 계약 판정\n"
    "```text\n"
    f"delta_version = {delta_table.version()}\n"
    f"partition_columns = {PARTITION_COLUMNS}\n"
    f"natural_key = {NATURAL_KEY_COLUMNS}\n"
    f"schema_status = {status_text}\n"
    "```"
))
display(schema_contract_df.sort_values(["status", "column_name"]))

### Raw schema 계약 판정
```text
delta_version = 0
partition_columns = ['block_date_utc']
natural_key = ('chain_id', 'transaction_hash', 'log_index')
schema_status = PARTIALLY VERIFIED
```

,column_name,expected_type,actual_type,expected_nullable,actual_nullable,status
0,address,NaN,string,None,False,EXTRA_IN_DATA
1,block_date,NaN,date32[day],None,False,EXTRA_IN_DATA
5,block_timestamp,NaN,"timestamp[us, tz=UTC]",None,False,EXTRA_IN_DATA
9,data,NaN,string,None,False,EXTRA_IN_DATA
13,ingested_at,NaN,"timestamp[us, tz=UTC]",None,False,EXTRA_IN_DATA
19,source_run_id,NaN,string,None,False,EXTRA_IN_DATA
2,block_date_utc,date32[day],NaN,False,None,MISSING_IN_DATA
6,block_timestamp_utc,"timestamp[us, tz=UTC]",NaN,False,None,MISSING_IN_DATA
8,contract_address,string,NaN,False,None,MISSING_IN_DATA
10,data_raw,string,NaN,False,None,MISSING_IN_DATA


## 2. Raw Delta row count, 중복, 시간 범위 확인

실제 Delta schema가 최신 계약과 다르더라도 row count와 natural key 중복은 확인합니다. 컬럼명이 구 schema인 경우에는 가능한 alias를 사용하고, 계약 불일치는 최종 판정에서 `PARTIALLY VERIFIED`로 남깁니다.

In [3]:
raw_dataset = delta_table.to_pyarrow_dataset()
raw_con = duckdb.connect()
raw_con.register("raw_logs", raw_dataset)
actual_columns = set(actual_by_name)

block_timestamp_column = "block_timestamp_utc" if "block_timestamp_utc" in actual_columns else "block_timestamp"
ingested_at_column = "ingested_at_utc" if "ingested_at_utc" in actual_columns else "ingested_at"
contract_column = "contract_address" if "contract_address" in actual_columns else "address"
data_column = "data_raw" if "data_raw" in actual_columns else "data"
interval_start_column = "interval_start_utc" if "interval_start_utc" in actual_columns else None
interval_end_column = "interval_end_utc" if "interval_end_utc" in actual_columns else None

raw_count = raw_con.execute("select count(*) from raw_logs").fetchone()[0]
raw_duplicate_key_count = raw_con.execute(
    f"""
    select count(*)
    from (
        select {', '.join(NATURAL_KEY_COLUMNS)}
        from raw_logs
        group by {', '.join(str(index) for index in range(1, len(NATURAL_KEY_COLUMNS) + 1))}
        having count(*) > 1
    )
    """
).fetchone()[0]

raw_summary_sql = f"""
select
    count(*) as raw_row_count,
    count(distinct transaction_hash) as transaction_count,
    min(block_number) as min_block_number,
    max(block_number) as max_block_number,
    min({block_timestamp_column}) as first_block_timestamp_utc,
    max({block_timestamp_column}) as latest_block_timestamp_utc,
    max({ingested_at_column}) as latest_ingested_at_utc
from raw_logs
"""
raw_summary_df = raw_con.execute(raw_summary_sql).fetchdf()
for column in ["first_block_timestamp_utc", "latest_block_timestamp_utc", "latest_ingested_at_utc"]:
    raw_summary_df = add_kst_column(raw_summary_df, column)

raw_health_df = pd.DataFrame(
    [
        {
            "check": "raw row count",
            "actual": raw_count,
            "expected": "> 0",
            "status": "PASS" if raw_count > 0 else "FAIL",
        },
        {
            "check": "raw natural key duplicate",
            "actual": raw_duplicate_key_count,
            "expected": 0,
            "status": "PASS" if raw_duplicate_key_count == 0 else "FAIL",
        },
        {
            "check": "raw schema matches current Python source",
            "actual": status_text,
            "expected": "VERIFIED",
            "status": "PASS" if RAW_SCHEMA_CURRENT else "PARTIALLY VERIFIED",
        },
    ]
)

display(raw_health_df)
display(raw_summary_df)

,check,actual,expected,status
0,raw row count,1,> 0,PASS
1,raw natural key duplicate,0,0,PASS
2,raw schema matches current Python source,PARTIALLY VERIFIED,VERIFIED,PARTIALLY VERIFIED


,raw_row_count,transaction_count,min_block_number,max_block_number,first_block_timestamp_utc,latest_block_timestamp_utc,latest_ingested_at_utc,first_block_timestamp_kst,latest_block_timestamp_kst,latest_ingested_at_kst
0,1,1,100,100,2024-01-01 00:00:00+00:00,2024-01-01 00:00:00+00:00,2024-01-01 01:00:00+00:00,2024-01-01 09:00:00+09:00,2024-01-01 09:00:00+09:00,2024-01-01 10:00:00+09:00


## 3. Raw sample 확인

최신 block timestamp 기준으로 raw log 샘플을 확인합니다. 구 schema와 신 schema가 섞여 있어도 샘플의 주요 식별자와 payload 흐름을 볼 수 있도록 공통 alias를 사용합니다.

In [4]:
sample_columns = [
    "chain_id",
    "block_number",
    f"{block_timestamp_column} as block_timestamp_utc",
    "transaction_hash",
    "log_index",
    f"{contract_column} as contract_address",
    "topic0",
    "topic1",
    "topic2",
    "topic3",
    f"{data_column} as data_raw",
]
if interval_start_column and interval_end_column:
    sample_columns.extend([
        f"{interval_start_column} as interval_start_utc",
        f"{interval_end_column} as interval_end_utc",
    ])
if "data_uint256_decimal_text" in actual_columns:
    sample_columns.append("data_uint256_decimal_text")
if "data_uint256_decode_status" in actual_columns:
    sample_columns.append("data_uint256_decode_status")

latest_raw_sample_df = raw_con.execute(
    f"""
    select
        {', '.join(sample_columns)}
    from raw_logs
    order by {block_timestamp_column} desc, log_index desc
    limit 20
    """
).fetchdf()

for column in ["block_timestamp_utc", "interval_start_utc", "interval_end_utc"]:
    latest_raw_sample_df = add_kst_column(latest_raw_sample_df, column)

display(latest_raw_sample_df)

,chain_id,block_number,block_timestamp_utc,transaction_hash,log_index,contract_address,topic0,topic1,topic2,topic3,data_raw,block_timestamp_kst
0,1,100,2024-01-01 00:00:00+00:00,0xbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb...,1,0xdac17f958d2ee523a2206206994597c13d831ec7,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x00000000000000000000000011111111111111111111...,0x0000000000000000000000005754284f345afc66a98f...,None,0x00000000000000000000000000000000000000000000...,2024-01-01 09:00:00+09:00


## 4. DuckDB downstream relation 상태 확인

`erc20_transfers`와 `tether_treasury_flow`는 분석 산출물의 핵심 relation입니다. `ethereum_logs` staging relation이 없고 legacy `stg_ethereum_logs`만 남아 있으면 최신 dbt naming으로 rebuild되지 않은 상태로 표시합니다.

In [5]:
analytics_con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
relations_df = analytics_con.execute(
    """
    select table_schema, table_name, table_type
    from information_schema.tables
    where table_schema = 'main'
    order by table_name
    """
).fetchdf()
relation_names = set(relations_df["table_name"])
display(relations_df)

expected_relations = ["ethereum_logs", "erc20_transfers", "tether_treasury_flow"]
relation_status_rows: list[dict[str, Any]] = []
for relation in expected_relations:
    exists = relation in relation_names
    count_value = None
    query_error = None
    if exists:
        count_value, query_error = safe_scalar(analytics_con, f"select count(*) from main.{relation}")
    relation_status_rows.append(
        {
            "relation": relation,
            "expected_role": "staging" if relation == "ethereum_logs" else "downstream",
            "exists": exists,
            "row_count": count_value,
            "query_error": query_error,
            "status": "PASS" if exists and query_error is None else "PARTIALLY VERIFIED",
        }
    )

if "ethereum_logs" not in relation_names and "stg_ethereum_logs" in relation_names:
    relation_status_rows.append(
        {
            "relation": "stg_ethereum_logs",
            "expected_role": "legacy staging name detected",
            "exists": True,
            "row_count": None,
            "query_error": "current dbt model name is ethereum_logs; existing DuckDB file still has legacy relation",
            "status": "PARTIALLY VERIFIED",
        }
    )

relation_status_df = pd.DataFrame(relation_status_rows)
display(relation_status_df)

,table_schema,table_name,table_type
0,main,erc20_transfer_integrity,VIEW
1,main,erc20_transfers,BASE TABLE
2,main,stg_ethereum_logs,VIEW
3,main,tether_treasury_flow,BASE TABLE
4,main,treasury_flow_integrity,VIEW
5,main,unique_log_identity,VIEW


,relation,expected_role,exists,row_count,query_error,status
0,ethereum_logs,staging,False,NaN,NaN,PARTIALLY VERIFIED
1,erc20_transfers,downstream,True,1.0,NaN,PASS
2,tether_treasury_flow,downstream,True,1.0,NaN,PASS
3,stg_ethereum_logs,legacy staging name detected,True,NaN,current dbt model name is ethereum_logs; exist...,PARTIALLY VERIFIED


## 5. Silver/Gold schema와 row count 확인

downstream relation이 존재하면 schema와 row count를 확인합니다. `DESCRIBE` 또는 count query 실패는 노트북 실행 실패로 숨기지 않고 표의 query error로 남깁니다.

In [6]:
downstream_schema_frames: list[pd.DataFrame] = []
downstream_count_rows: list[dict[str, Any]] = []
for relation in ["erc20_transfers", "tether_treasury_flow"]:
    if relation not in relation_names:
        downstream_count_rows.append(
            {
                "relation": relation,
                "row_count": None,
                "duplicate_natural_key_count": None,
                "latest_block_timestamp_utc": None,
                "status": "PARTIALLY VERIFIED",
                "query_error": "relation missing",
            }
        )
        continue

    schema_df, schema_error = safe_execute_df(analytics_con, f"describe main.{relation}")
    if schema_df is not None:
        schema_df.insert(0, "relation", relation)
        downstream_schema_frames.append(schema_df)

    row_count, row_error = safe_scalar(analytics_con, f"select count(*) from main.{relation}")
    describe_columns = set(schema_df["column_name"]) if schema_df is not None and "column_name" in schema_df.columns else set()
    ts_column = "block_timestamp_utc" if "block_timestamp_utc" in describe_columns else "block_timestamp" if "block_timestamp" in describe_columns else None

    latest_ts = None
    latest_ts_error = None
    if ts_column:
        latest_ts, latest_ts_error = safe_scalar(analytics_con, f"select max({ts_column}) from main.{relation}")

    duplicate_count = None
    duplicate_error = None
    if all(column in describe_columns for column in NATURAL_KEY_COLUMNS):
        duplicate_count, duplicate_error = safe_scalar(
            analytics_con,
            f"""
            select count(*)
            from (
                select {', '.join(NATURAL_KEY_COLUMNS)}
                from main.{relation}
                group by {', '.join(str(index) for index in range(1, len(NATURAL_KEY_COLUMNS) + 1))}
                having count(*) > 1
            )
            """,
        )

    query_error = schema_error or row_error or latest_ts_error or duplicate_error
    downstream_count_rows.append(
        {
            "relation": relation,
            "row_count": row_count,
            "duplicate_natural_key_count": duplicate_count,
            "latest_block_timestamp_utc": latest_ts,
            "latest_block_timestamp_kst": as_utc_timestamp(latest_ts).tz_convert(DISPLAY_TZ) if latest_ts is not None else None,
            "status": "PASS" if query_error is None and (duplicate_count in (0, None)) else "PARTIALLY VERIFIED",
            "query_error": query_error,
        }
    )

downstream_count_df = pd.DataFrame(downstream_count_rows)
display(downstream_count_df)
if downstream_schema_frames:
    display(pd.concat(downstream_schema_frames, ignore_index=True))

,relation,row_count,duplicate_natural_key_count,latest_block_timestamp_utc,latest_block_timestamp_kst,status,query_error
0,erc20_transfers,1,0.0,2024-01-01 09:00:00,2024-01-01 18:00:00+09:00,PASS,None
1,tether_treasury_flow,1,NaN,NaT,NaT,PASS,None


,relation,column_name,column_type,null,key,default,extra
0,erc20_transfers,chain_id,BIGINT,YES,None,None,None
1,erc20_transfers,block_number,BIGINT,YES,None,None,None
2,erc20_transfers,block_timestamp,TIMESTAMP,YES,None,None,None
3,erc20_transfers,block_date,DATE,YES,None,None,None
4,erc20_transfers,transaction_hash,VARCHAR,YES,None,None,None
5,erc20_transfers,log_index,BIGINT,YES,None,None,None
6,erc20_transfers,token_address,VARCHAR,YES,None,None,None
7,erc20_transfers,from_address,VARCHAR,YES,None,None,None
8,erc20_transfers,to_address,VARCHAR,YES,None,None,None
9,erc20_transfers,amount_raw_hex,VARCHAR,YES,None,None,None


## 6. Raw ↔ Silver 최신성 비교

raw와 silver가 모두 조회 가능할 때 최신 event timestamp를 비교합니다. 다만 raw schema가 현재 Python source 계약과 다르면 downstream 결과가 과거 빌드 산출물일 수 있으므로 최종 상태는 `PARTIALLY VERIFIED`로 유지합니다.

In [7]:
raw_latest_ts = as_utc_timestamp(raw_summary_df.loc[0, "latest_block_timestamp_utc"])

silver_latest_ts = None
silver_error = None
if "erc20_transfers" in relation_names:
    erc20_schema_df, schema_error = safe_execute_df(analytics_con, "describe main.erc20_transfers")
    erc20_columns = set(erc20_schema_df["column_name"]) if erc20_schema_df is not None else set()
    silver_ts_column = "block_timestamp_utc" if "block_timestamp_utc" in erc20_columns else "block_timestamp" if "block_timestamp" in erc20_columns else None
    if silver_ts_column:
        silver_latest_ts_raw, silver_error = safe_scalar(analytics_con, f"select max({silver_ts_column}) from main.erc20_transfers")
        silver_latest_ts = as_utc_timestamp(silver_latest_ts_raw)
    else:
        silver_error = schema_error or "erc20_transfers has no block_timestamp column"
else:
    silver_error = "erc20_transfers relation missing"

if raw_latest_ts is not None and silver_latest_ts is not None:
    lag = raw_latest_ts - silver_latest_ts
    freshness_status = "PASS" if silver_latest_ts >= raw_latest_ts else "PARTIALLY VERIFIED"
else:
    lag = None
    freshness_status = "PARTIALLY VERIFIED"

freshness_df = pd.DataFrame(
    [
        {
            "check": "raw latest event timestamp",
            "utc": raw_latest_ts,
            "kst": raw_latest_ts.tz_convert(DISPLAY_TZ) if raw_latest_ts is not None else None,
            "status": "INFO",
            "detail": None,
        },
        {
            "check": "silver latest event timestamp",
            "utc": silver_latest_ts,
            "kst": silver_latest_ts.tz_convert(DISPLAY_TZ) if silver_latest_ts is not None else None,
            "status": "INFO" if silver_error is None else "PARTIALLY VERIFIED",
            "detail": silver_error,
        },
        {
            "check": "raw to silver freshness",
            "utc": None,
            "kst": None,
            "status": freshness_status,
            "detail": f"lag={lag}" if lag is not None else "timestamp comparison unavailable",
        },
    ]
)
display(freshness_df)

,check,utc,kst,status,detail
0,raw latest event timestamp,2024-01-01 00:00:00+00:00,2024-01-01 09:00:00+09:00,INFO,NaN
1,silver latest event timestamp,2024-01-01 09:00:00+00:00,2024-01-01 18:00:00+09:00,INFO,NaN
2,raw to silver freshness,NaT,NaT,PASS,lag=-1 days +15:00:00


## 7. 최종 판정

이 판정은 노트북 실행 시점의 로컬 산출물 기준입니다. 외부 RPC Provider의 지속 운영 안정성 검증으로 해석하지 않습니다.

In [8]:
hard_failures = []
partial_reasons = []

if raw_count <= 0:
    hard_failures.append("raw Delta row count is zero")
if raw_duplicate_key_count != 0:
    hard_failures.append("raw Delta natural key duplicates exist")
if not RAW_SCHEMA_CURRENT:
    partial_reasons.append("actual raw Delta schema does not match current Python delta_writer contract")

required_downstream = relation_status_df[relation_status_df["relation"].isin(["erc20_transfers", "tether_treasury_flow"])]
if not required_downstream["exists"].all():
    hard_failures.append("required downstream relation missing")
if required_downstream["query_error"].notna().any():
    partial_reasons.append("one or more downstream relation queries returned an error")

if "ethereum_logs" not in relation_names:
    partial_reasons.append("current staging relation ethereum_logs is missing from DuckDB file")
if freshness_status != "PASS":
    partial_reasons.append("raw to silver freshness is not fully verified")

if hard_failures:
    final_status = "NOT VERIFIED"
elif partial_reasons:
    final_status = "PARTIALLY VERIFIED"
else:
    final_status = "VERIFIED"

final_status_df = pd.DataFrame(
    [
        {
            "area": "canonical raw Delta",
            "status": "VERIFIED" if RAW_SCHEMA_CURRENT and raw_duplicate_key_count == 0 and raw_count > 0 else "PARTIALLY VERIFIED",
            "evidence": f"rows={raw_count}, duplicate_keys={raw_duplicate_key_count}, schema_current={RAW_SCHEMA_CURRENT}",
        },
        {
            "area": "dbt downstream relations",
            "status": "VERIFIED" if required_downstream["exists"].all() and required_downstream["query_error"].isna().all() else "PARTIALLY VERIFIED",
            "evidence": relation_status_df.to_dict(orient="records"),
        },
        {
            "area": "raw to silver freshness",
            "status": "VERIFIED" if freshness_status == "PASS" and RAW_SCHEMA_CURRENT else "PARTIALLY VERIFIED",
            "evidence": freshness_df.to_dict(orient="records"),
        },
    ]
)

display(Markdown(
    "### Notebook validation result\n"
    "```text\n"
    f"final_status = {final_status}\n"
    f"hard_failures = {hard_failures}\n"
    f"partial_reasons = {partial_reasons}\n"
    "```"
))
display(final_status_df)

### Notebook validation result
```text
final_status = PARTIALLY VERIFIED
hard_failures = []
partial_reasons = ['actual raw Delta schema does not match current Python delta_writer contract', 'current staging relation ethereum_logs is missing from DuckDB file']
```

,area,status,evidence
0,canonical raw Delta,PARTIALLY VERIFIED,"rows=1, duplicate_keys=0, schema_current=False"
1,dbt downstream relations,VERIFIED,"[{'relation': 'ethereum_logs', 'expected_role'..."
2,raw to silver freshness,PARTIALLY VERIFIED,"[{'check': 'raw latest event timestamp', 'utc'..."
